## Notebook con el proyecto final del grupo Condores

## DataFrame — Estados de licitación y etapas.
Esta información es fundamental para entender los diferentes estados en los que una licitación puede encontrarse, más adelante veremos que únicamente trabajaremos con licitaciones de un tipo en particular de estos.

In [17]:
# =========================================================
# CATALOGO OFICIAL ESTADOS DE LICITACION (CHILECOMPRA API)
# =========================================================

import pandas as pd

data_estados = {
    "CodigoEstado": [
        5, 6, 7, 8, 18, 19
    ],

    "Estado": [
        "Publicada",
        "Cerrada",
        "Desierta",
        "Adjudicada",
        "Revocada",
        "Suspendida"
    ],

    "Descripcion": [
        "Licitación publicada y abierta a recepción de ofertas",
        "Licitación cerrada (fin de recepción de ofertas)",
        "Licitación sin ofertas válidas",
        "Licitación adjudicada a un proveedor",
        "Licitación revocada por el organismo comprador",
        "Licitación suspendida temporalmente"
    ],

    # 👇 MUY IMPORTANTE PARA ANALITICA
    "GrupoEstado": [
        "Activa",
        "Cerrada",
        "Cerrada",
        "Cerrada",
        "Cerrada",
        "Activa"
    ]
}

df_estados_licitacion = pd.DataFrame(data_estados)

display(df_estados_licitacion)

,CodigoEstado,Estado,Descripcion,GrupoEstado
0,5,Publicada,Licitación publicada y abierta a recepción de ...,Activa
1,6,Cerrada,Licitación cerrada (fin de recepción de ofertas),Cerrada
2,7,Desierta,Licitación sin ofertas válidas,Cerrada
3,8,Adjudicada,Licitación adjudicada a un proveedor,Cerrada
4,18,Revocada,Licitación revocada por el organismo comprador,Cerrada
5,19,Suspendida,Licitación suspendida temporalmente,Activa


## Tipos de Licitación: Es complementaria al estado y nos indica los diferentes tipos de licitaciones que pueden existir, igual es relevante esta información porque más adelante trabajremos únicamente con un tipo de licitación en particular

In [18]:
# =========================================================
# CATALOGO OFICIAL TIPOS DE LICITACION (CHILECOMPRA)
# =========================================================

import pandas as pd

data_tipos = {
    "codigo_tipo": [
        "L1", "LE", "LP", "LQ", "LR",
        "E2", "CO", "B2", "H2", "I2",
        "LS"
    ],

    "tipo_licitacion": [
        "Licitación Pública Menor a 100 UTM",
        "Licitación Pública entre 100 y 1.000 UTM",
        "Licitación Pública entre 1.000 y 2.000 UTM",
        "Licitación Pública entre 2.000 y 5.000 UTM",
        "Licitación Pública Mayor a 5.000 UTM",
        "Licitación Privada Menor a 100 UTM",
        "Licitación Privada entre 100 y 1.000 UTM",
        "Licitación Privada entre 1.000 y 2.000 UTM",
        "Licitación Privada entre 2.000 y 5.000 UTM",
        "Licitación Privada Mayor a 5.000 UTM",
        "Licitación Servicios Personales Especializados"
    ],

    # 👇 CLAVE PARA TU PROYECTO
    "tipo_base": [
        "Publica","Publica","Publica","Publica","Publica",
        "Privada","Privada","Privada","Privada","Privada",
        "Especial"
    ],

    # 👇 SEGMENTACIÓN POR MONTO (MUY UTIL PARA ML)
    "rango_monto": [
        "<100 UTM",
        "100-1000 UTM",
        "1000-2000 UTM",
        "2000-5000 UTM",
        ">5000 UTM",
        "<100 UTM",
        "100-1000 UTM",
        "1000-2000 UTM",
        "2000-5000 UTM",
        ">5000 UTM",
        "N/A"
    ]
}

df_tipos_licitacion = pd.DataFrame(data_tipos)

display(df_tipos_licitacion)

,codigo_tipo,tipo_licitacion,tipo_base,rango_monto
0,L1,Licitación Pública Menor a 100 UTM,Publica,<100 UTM
1,LE,Licitación Pública entre 100 y 1.000 UTM,Publica,100-1000 UTM
2,LP,Licitación Pública entre 1.000 y 2.000 UTM,Publica,1000-2000 UTM
3,LQ,Licitación Pública entre 2.000 y 5.000 UTM,Publica,2000-5000 UTM
4,LR,Licitación Pública Mayor a 5.000 UTM,Publica,>5000 UTM
5,E2,Licitación Privada Menor a 100 UTM,Privada,<100 UTM
6,CO,Licitación Privada entre 100 y 1.000 UTM,Privada,100-1000 UTM
7,B2,Licitación Privada entre 1.000 y 2.000 UTM,Privada,1000-2000 UTM
8,H2,Licitación Privada entre 2.000 y 5.000 UTM,Privada,2000-5000 UTM
9,I2,Licitación Privada Mayor a 5.000 UTM,Privada,>5000 UTM


## 📦 Código — Paso 1: Carga dataset

Lectura de la información
Como las fuentes de datos son bastante pesadas 
    Licitaciones 42.3MB
se toma la decisión de subir las fuentes a google Drive y desde allí gestionar los permisos de descarga de la información

In [2]:
import os
import requests
import pandas as pd

# Tu nuevo ID de archivo
FILE_ID = '11fA5-HKOX0YmWflZdRFmuK7wKrzX1f05'
URL = f'https://drive.google.com/uc?export=download&id={FILE_ID}'
NOMBRE_LOCAL = 'licitaciones_final.parquet'

print("--- DESCARGANDO DESDE TU DRIVE ---")

try:
    # Descarga directa
    response = requests.get(URL, stream=True)
    with open(NOMBRE_LOCAL, "wb") as f:
        for chunk in response.iter_content(chunk_size=32768):
            if chunk: f.write(chunk)
            
    # Verificación de integridad
    peso_mb = os.path.getsize(NOMBRE_LOCAL) / (1024 * 1024)
    print(f"Archivo descargado: {peso_mb:.2f} MB")
    
    if peso_mb > 1.0:
        df_licitaciones = pd.read_parquet(NOMBRE_LOCAL, engine='pyarrow')
        print(f"✅ ¡ÉXITO! Licitaciones cargadas: {df_licitaciones.shape} filas.")
        display(df_licitaciones.head())
    else:
        print("❌ El archivo sigue siendo demasiado pequeño. Revisa que el permiso sea 'Cualquier persona con el enlace'.")

except Exception as e:
    print(f"❌ Error: {e}")


--- DESCARGANDO DESDE TU DRIVE ---
Archivo descargado: 42.33 MB
✅ ¡ÉXITO! Licitaciones cargadas: (826238, 4) filas.


,CodigoExterno,Nombre,CodigoEstado,FechaCierre
0,564162-187-L119,(id.48430) Particulas para Depto de Física,7,2020-01-01T17:04:01.527
1,1058134-381-L119,Medicamentos Enero 2020,7,2020-01-01T15:04:01.643
2,1663-111-L119,SERVICIO DE LABORATORIO DENTAL PARA PRÓTESIS M...,7,2020-01-01T21:44:01.157
3,2713-290-L119,CLASES DIRIGIDAS,7,2020-01-01T19:29:01.22
4,1058133-15-LR19,Convenio de Suministro Farmacos Inyectables Red,8,2019-11-04T15:01:00


In [4]:
# Columnas del dataset
print("\n==============================")
print("COLUMNAS")
print("==============================")

print(df_licitaciones.columns.tolist())


COLUMNAS
['CodigoExterno', 'Nombre', 'CodigoEstado', 'FechaCierre']


In [3]:
# Tipos de datos

print("\n==============================")
print("TIPOS DE DATOS")
print("==============================")

display(df_licitaciones.dtypes)


TIPOS DE DATOS


CodigoExterno    object
Nombre           object
CodigoEstado      int64
FechaCierre      object
dtype: object

In [5]:
# Información general del Dataset

df_licitaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 826238 entries, 0 to 826237
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   CodigoExterno  826238 non-null  object
 1   Nombre         826238 non-null  object
 2   CodigoEstado   826238 non-null  int64 
 3   FechaCierre    826237 non-null  object
dtypes: int64(1), object(3)
memory usage: 25.2+ MB


In [6]:
# Valores nulos por columna

nulls = df_licitaciones.isnull().sum().sort_values(ascending=False)

display(nulls.head(20))

FechaCierre      1
CodigoExterno    0
Nombre           0
CodigoEstado     0
dtype: int64

In [7]:
# Registros duplicados

duplicados = df_licitaciones.duplicated().sum()

print("Total duplicados:", duplicados)


Total duplicados: 0


In [9]:
# Uso de memoria

memoria = df_licitaciones.memory_usage(deep=True).sum() / 1024**2

print(f"Memoria utilizada: {memoria:.2f} MB")

Memoria utilizada: 228.47 MB


In [10]:
# Último vistazo al dataframe

df_licitaciones.head()

,CodigoExterno,Nombre,CodigoEstado,FechaCierre
0,564162-187-L119,(id.48430) Particulas para Depto de Física,7,2020-01-01T17:04:01.527
1,1058134-381-L119,Medicamentos Enero 2020,7,2020-01-01T15:04:01.643
2,1663-111-L119,SERVICIO DE LABORATORIO DENTAL PARA PRÓTESIS M...,7,2020-01-01T21:44:01.157
3,2713-290-L119,CLASES DIRIGIDAS,7,2020-01-01T19:29:01.22
4,1058133-15-LR19,Convenio de Suministro Farmacos Inyectables Red,8,2019-11-04T15:01:00


## 📦 Código — Paso 2: Embudo Licitaciones.

Este paso es fundamental en nuestro proyecto ya que únicamente vamos a trabajar con licitaciones de tipo ***"adjudicadas"*** que en el campo CodigoEstado corresponden al estado número 8

In [11]:
# Filtrar licitaciones adjudicadas

df_licitaciones_adjudicadas = df_licitaciones[
    df_licitaciones["CodigoEstado"] == 8
].copy()

print("Total licitaciones adjudicadas:", len(df_licitaciones_adjudicadas))

display(df_licitaciones_adjudicadas.head())

Total licitaciones adjudicadas: 604561


,CodigoExterno,Nombre,CodigoEstado,FechaCierre
4,1058133-15-LR19,Convenio de Suministro Farmacos Inyectables Red,8,2019-11-04T15:01:00
5,1058133-19-LR19,Conv.Sum Farmacos trastornos Cardiovasculares RED,8,2019-11-11T15:01:00
8,1523-97-LP19,Propuesta Pública Nº 101/19 “SUMINISTRO DE INS...,8,2019-12-02T15:30:00
9,2097-88-LE19,MEDICAMENTO SOLICITUD DE COMPRA SIABI N 4270 -...,8,2019-12-12T16:00:00
13,2436-575-LE19,Contrato de Suministro Servicio de arriendo am...,8,2019-12-26T16:00:00


In [12]:
df_licitaciones_adjudicadas.info()

<class 'pandas.core.frame.DataFrame'>
Index: 604561 entries, 4 to 826237
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   CodigoExterno  604561 non-null  object
 1   Nombre         604561 non-null  object
 2   CodigoEstado   604561 non-null  int64 
 3   FechaCierre    604561 non-null  object
dtypes: int64(1), object(3)
memory usage: 23.1+ MB


El siguiente paso consiste en descomponer el CodigoExterno, el cual internamente contiene información relevante y necesaria

In [13]:
# =========================================================
# DESCOMPONER CodigoExterno COMPLETO
# =========================================================

df = df_licitaciones_adjudicadas.copy()

# separar partes
df[["codigo_organismo", "correlativo", "tipo_anio"]] = (
    df["CodigoExterno"].str.split("-", expand=True)
)

# tipo
df["codigo_tipo"] = df["tipo_anio"].str.replace(r"\d+", "", regex=True)

# año
df["anio_licitacion"] = df["tipo_anio"].str.extract(r"(\d+)").astype('Int64') # Changed to 'Int64' to handle NaN values

display(df.head())

,CodigoExterno,Nombre,CodigoEstado,FechaCierre,codigo_organismo,correlativo,tipo_anio,codigo_tipo,anio_licitacion
4,1058133-15-LR19,Convenio de Suministro Farmacos Inyectables Red,8,2019-11-04T15:01:00,1058133,15,LR19,LR,19
5,1058133-19-LR19,Conv.Sum Farmacos trastornos Cardiovasculares RED,8,2019-11-11T15:01:00,1058133,19,LR19,LR,19
8,1523-97-LP19,Propuesta Pública Nº 101/19 “SUMINISTRO DE INS...,8,2019-12-02T15:30:00,1523,97,LP19,LP,19
9,2097-88-LE19,MEDICAMENTO SOLICITUD DE COMPRA SIABI N 4270 -...,8,2019-12-12T16:00:00,2097,88,LE19,LE,19
13,2436-575-LE19,Contrato de Suministro Servicio de arriendo am...,8,2019-12-26T16:00:00,2436,575,LE19,LE,19


In [14]:
# Verificamos que con la descomposición no hayamos perdidos registros

df["CodigoEstado"].value_counts()

CodigoEstado
8    604561
Name: count, dtype: int64

In [15]:
# =========================================================
# EXTRAER CODIGO TIPO CORRECTAMENTE
# =========================================================

df["tipo_raw"] = df["CodigoExterno"].str.split("-").str[-1]

# tomar SOLO los primeros 2 caracteres
df["codigo_tipo"] = df["tipo_raw"].str[:2]

display(df[["CodigoExterno", "tipo_raw", "codigo_tipo"]].head())

,CodigoExterno,tipo_raw,codigo_tipo
4,1058133-15-LR19,LR19,LR
5,1058133-19-LR19,LR19,LR
8,1523-97-LP19,LP19,LP
9,2097-88-LE19,LE19,LE
13,2436-575-LE19,LE19,LE


Luego revisamos los tipos de licitaciones que más cantidad de adjudicaciones tuvieron, este ranking es clave porque nos permite visualizar las ordenes con la que más adelante trabajaremos

In [16]:
# Ranging por tipo de licitación

ranking_tipos = (
    df.groupby("codigo_tipo")
    .size()
    .reset_index(name="cantidad_adjudicadas")
    .sort_values(by="cantidad_adjudicadas", ascending=False)
    .reset_index(drop=True)
)

ranking_tipos["ranking"] = ranking_tipos.index + 1

display(ranking_tipos)

print("Suma total:", ranking_tipos["cantidad_adjudicadas"].sum())

,codigo_tipo,cantidad_adjudicadas,ranking
0,LE,266632,1
1,L1,199890,2
2,LP,44798,3
3,LQ,44144,4
4,LR,24585,5
5,R1,13293,6
6,O1,6181,7
7,CO,2378,8
8,E2,842,9
9,R2,581,10


Suma total: 604561


In [19]:
# Recordemos el df de los tipos de licitaciones
df_tipos_licitacion.head()

,codigo_tipo,tipo_licitacion,tipo_base,rango_monto
0,L1,Licitación Pública Menor a 100 UTM,Publica,<100 UTM
1,LE,Licitación Pública entre 100 y 1.000 UTM,Publica,100-1000 UTM
2,LP,Licitación Pública entre 1.000 y 2.000 UTM,Publica,1000-2000 UTM
3,LQ,Licitación Pública entre 2.000 y 5.000 UTM,Publica,2000-5000 UTM
4,LR,Licitación Pública Mayor a 5.000 UTM,Publica,>5000 UTM


Vamos a agregar un clasificación adicional a las licitaciones, si son públicas, privadas o especiales

In [20]:
# =========================================================
# AGREGAR TIPO BASE (PUBLICA / PRIVADA / ESPECIAL)
# =========================================================

# Clean up potentially existing suffixed columns from previous runs
columns_to_drop = [col for col in ['tipo_base_x', 'tipo_base_y', 'tipo_base'] if col in df.columns]
if columns_to_drop:
    df = df.drop(columns=columns_to_drop)

df = df.merge(
    df_tipos_licitacion[["codigo_tipo", "tipo_base"]],
    on="codigo_tipo",
    how="left"
)

In [21]:
display(df.head())

,CodigoExterno,Nombre,CodigoEstado,FechaCierre,codigo_organismo,correlativo,tipo_anio,codigo_tipo,anio_licitacion,tipo_raw,tipo_base
0,1058133-15-LR19,Convenio de Suministro Farmacos Inyectables Red,8,2019-11-04T15:01:00,1058133,15,LR19,LR,19,LR19,Publica
1,1058133-19-LR19,Conv.Sum Farmacos trastornos Cardiovasculares RED,8,2019-11-11T15:01:00,1058133,19,LR19,LR,19,LR19,Publica
2,1523-97-LP19,Propuesta Pública Nº 101/19 “SUMINISTRO DE INS...,8,2019-12-02T15:30:00,1523,97,LP19,LP,19,LP19,Publica
3,2097-88-LE19,MEDICAMENTO SOLICITUD DE COMPRA SIABI N 4270 -...,8,2019-12-12T16:00:00,2097,88,LE19,LE,19,LE19,Publica
4,2436-575-LE19,Contrato de Suministro Servicio de arriendo am...,8,2019-12-26T16:00:00,2436,575,LE19,LE,19,LE19,Publica


Con esta nueva clasificación vamos a revisar cuales tipos (pública, privada, especial) de licitación son las que más se realizan

In [22]:
# =========================================================
# PORCENTAJE DE CADA TIPO DE BASE
# =========================================================

tabla_tipo_base = (
    df
    .groupby("tipo_base")
    .size()
    .reset_index(name="cantidad_licitaciones")
)

total_licitaciones = tabla_tipo_base["cantidad_licitaciones"].sum()

tabla_tipo_base["porcentaje"] = (
    tabla_tipo_base["cantidad_licitaciones"] / total_licitaciones * 100
)

tabla_tipo_base = tabla_tipo_base.sort_values("cantidad_licitaciones", ascending=False).reset_index(drop=True)

display(tabla_tipo_base)

,tipo_base,cantidad_licitaciones,porcentaje
0,Publica,580049,99.259212
1,Privada,4220,0.722135
2,Especial,109,0.018652


El resultado es contundente en que las liciaciones publicas representan el 99.2% del total de licitaciones, por lo tanto, únicamente seguiremos trabajando con las "Publicas"

In [23]:
# =========================================================
# FILTRAR DATAFRAME POR TIPO DE BASE 'PUBLICAS'
# =========================================================

df_publicas = df[df["tipo_base"] == "Publica"].copy()

print("Total de licitaciones públicas:", len(df_publicas))
display(df_publicas.head())

Total de licitaciones públicas: 580049


,CodigoExterno,Nombre,CodigoEstado,FechaCierre,codigo_organismo,correlativo,tipo_anio,codigo_tipo,anio_licitacion,tipo_raw,tipo_base
0,1058133-15-LR19,Convenio de Suministro Farmacos Inyectables Red,8,2019-11-04T15:01:00,1058133,15,LR19,LR,19,LR19,Publica
1,1058133-19-LR19,Conv.Sum Farmacos trastornos Cardiovasculares RED,8,2019-11-11T15:01:00,1058133,19,LR19,LR,19,LR19,Publica
2,1523-97-LP19,Propuesta Pública Nº 101/19 “SUMINISTRO DE INS...,8,2019-12-02T15:30:00,1523,97,LP19,LP,19,LP19,Publica
3,2097-88-LE19,MEDICAMENTO SOLICITUD DE COMPRA SIABI N 4270 -...,8,2019-12-12T16:00:00,2097,88,LE19,LE,19,LE19,Publica
4,2436-575-LE19,Contrato de Suministro Servicio de arriendo am...,8,2019-12-26T16:00:00,2436,575,LE19,LE,19,LE19,Publica


In [24]:
df_publicas.info()

<class 'pandas.core.frame.DataFrame'>
Index: 580049 entries, 0 to 604559
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   CodigoExterno     580049 non-null  object
 1   Nombre            580049 non-null  object
 2   CodigoEstado      580049 non-null  int64 
 3   FechaCierre       580049 non-null  object
 4   codigo_organismo  580049 non-null  object
 5   correlativo       580049 non-null  object
 6   tipo_anio         580049 non-null  object
 7   codigo_tipo       580049 non-null  object
 8   anio_licitacion   580049 non-null  Int64 
 9   tipo_raw          580049 non-null  object
 10  tipo_base         580049 non-null  object
dtypes: Int64(1), int64(1), object(9)
memory usage: 53.7+ MB


Teniendo únicamente licitaciones públicas, ahora revisaremos el tipo "codigo_tipo" las cuales tienen que ver con el monto licitado "UTM"

In [25]:
# =========================================================
# PARTICIPACION (%) POR CODIGO_TIPO
# =========================================================

participacion_tipos = (
    df_publicas
    .groupby("codigo_tipo")
    .size()
    .reset_index(name="cantidad")
)

# Total
total = participacion_tipos["cantidad"].sum()

# Calcular %
participacion_tipos["participacion_%"] = (
    participacion_tipos["cantidad"] / total * 100
)

# Ordenar
participacion_tipos = participacion_tipos.sort_values(
    by="participacion_%", ascending=False
).reset_index(drop=True)

display(participacion_tipos)

,codigo_tipo,cantidad,participacion_%
0,LE,266632,45.967151
1,L1,199890,34.460882
2,LP,44798,7.723141
3,LQ,44144,7.610392
4,LR,24585,4.238435


Las licitaciones de tipo LE corresponden a: "Licitación Pública entre 100 y 1.000 UTM", como se ve representan el 46% del total de licitaciones, por lo que decidimos continuar trabajando con este tipo de licitaciones.

In [26]:
# =========================================================
# FILTRAR LE + AÑO ENTRE 20 Y 25
# =========================================================

df_publicas_le = df_publicas[
    (df_publicas["codigo_tipo"] == "LE") &
    (df_publicas["anio_licitacion"] >= 20) &
    (df_publicas["anio_licitacion"] <= 25)
].copy()

print("Total de licitaciones públicas tipo 'LE' (2020-2025):", len(df_publicas_le))
display(df_publicas_le.head())

Total de licitaciones públicas tipo 'LE' (2020-2025): 263454


,CodigoExterno,Nombre,CodigoEstado,FechaCierre,codigo_organismo,correlativo,tipo_anio,codigo_tipo,anio_licitacion,tipo_raw,tipo_base
1442,3568-2-LE20,CONVENIO SUMINISTRO DE MATERIALES DE FERRETERIA,8,2020-01-08T09:05:00,3568,2,LE20,LE,20,LE20,Publica
1481,525512-3-LE20,ADQUISICIÓN DE INSUMOS PARA PROGRAMA MUNICIPAL,8,2020-01-09T10:01:00,525512,3,LE20,LE,20,LE20,Publica
1700,3293-3-LE20,Administración Red Eléctrica Municipal Chaitén...,8,2020-01-08T09:00:00,3293,3,LE20,LE,20,LE20,Publica
1857,3796-1-LE20,Ferias Costumbristas verano 2020,8,2020-01-08T15:00:00,3796,1,LE20,LE,20,LE20,Publica
1905,525512-2-LE20,SERVICIO Y PRODUCCIÓN DE EVENTOS PARA PROGRAMA...,8,2020-01-09T20:00:00,525512,2,LE20,LE,20,LE20,Publica


In [58]:
df_publicas_le.info()

<class 'pandas.core.frame.DataFrame'>
Index: 263454 entries, 1442 to 604559
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   CodigoExterno     263454 non-null  object
 1   Nombre            263454 non-null  object
 2   CodigoEstado      263454 non-null  int64 
 3   FechaCierre       263454 non-null  object
 4   codigo_organismo  263454 non-null  object
 5   correlativo       263454 non-null  object
 6   tipo_anio         263454 non-null  object
 7   codigo_tipo       263454 non-null  object
 8   anio_licitacion   263454 non-null  Int64 
 9   tipo_raw          263454 non-null  object
 10  tipo_base         263454 non-null  object
dtypes: Int64(1), int64(1), object(9)
memory usage: 24.4+ MB


## 📦 Código — Paso 3: Analisis de Organismos.

In [59]:
df_publicas_le.head()

,CodigoExterno,Nombre,CodigoEstado,FechaCierre,codigo_organismo,correlativo,tipo_anio,codigo_tipo,anio_licitacion,tipo_raw,tipo_base
1442,3568-2-LE20,CONVENIO SUMINISTRO DE MATERIALES DE FERRETERIA,8,2020-01-08T09:05:00,3568,2,LE20,LE,20,LE20,Publica
1481,525512-3-LE20,ADQUISICIÓN DE INSUMOS PARA PROGRAMA MUNICIPAL,8,2020-01-09T10:01:00,525512,3,LE20,LE,20,LE20,Publica
1700,3293-3-LE20,Administración Red Eléctrica Municipal Chaitén...,8,2020-01-08T09:00:00,3293,3,LE20,LE,20,LE20,Publica
1857,3796-1-LE20,Ferias Costumbristas verano 2020,8,2020-01-08T15:00:00,3796,1,LE20,LE,20,LE20,Publica
1905,525512-2-LE20,SERVICIO Y PRODUCCIÓN DE EVENTOS PARA PROGRAMA...,8,2020-01-09T20:00:00,525512,2,LE20,LE,20,LE20,Publica


In [60]:
df_publicas_le.info()

<class 'pandas.core.frame.DataFrame'>
Index: 263454 entries, 1442 to 604559
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   CodigoExterno     263454 non-null  object
 1   Nombre            263454 non-null  object
 2   CodigoEstado      263454 non-null  int64 
 3   FechaCierre       263454 non-null  object
 4   codigo_organismo  263454 non-null  object
 5   correlativo       263454 non-null  object
 6   tipo_anio         263454 non-null  object
 7   codigo_tipo       263454 non-null  object
 8   anio_licitacion   263454 non-null  Int64 
 9   tipo_raw          263454 non-null  object
 10  tipo_base         263454 non-null  object
dtypes: Int64(1), int64(1), object(9)
memory usage: 24.4+ MB


Verificamos que los datos únicamente tengan licitaciones públicas y de tipo "LE" 

In [29]:
print(df_publicas_le['codigo_tipo'].unique())
print(df_publicas_le['tipo_base'].unique())

['LE']
['Publica']


In [64]:
conteo_org = df_publicas_le.groupby('codigo_organismo')['CodigoExterno'].count()

In [65]:
conteo_org.head()

codigo_organismo
1000       164
1000813    128
1001       125
1001048     14
1001546     10
Name: CodigoExterno, dtype: int64

In [72]:
conteo_org.info()


<class 'pandas.core.series.Series'>
Index: 4683 entries, 1000 to 999
Series name: CodigoExterno
Non-Null Count  Dtype
--------------  -----
4683 non-null   int64
dtypes: int64(1)
memory usage: 202.2+ KB


In [68]:
org_validos = conteo_org[conteo_org >= 150].index

In [71]:
org_validos.nunique()

451

In [54]:
conteo_org.info()

<class 'pandas.core.series.Series'>
Index: 4683 entries, 1000 to 999
Series name: CodigoExterno
Non-Null Count  Dtype
--------------  -----
4683 non-null   int64
dtypes: int64(1)
memory usage: 73.2+ KB


In [73]:
org_validos = conteo_org[conteo_org >= 150].index
df_filtrado = df_publicas_le[df_publicas_le['codigo_organismo'].isin(org_validos)]

In [74]:
df_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 128849 entries, 2180 to 604557
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   CodigoExterno     128849 non-null  object
 1   Nombre            128849 non-null  object
 2   CodigoEstado      128849 non-null  int64 
 3   FechaCierre       128849 non-null  object
 4   codigo_organismo  128849 non-null  object
 5   correlativo       128849 non-null  object
 6   tipo_anio         128849 non-null  object
 7   codigo_tipo       128849 non-null  object
 8   anio_licitacion   128849 non-null  Int64 
 9   tipo_raw          128849 non-null  object
 10  tipo_base         128849 non-null  object
dtypes: Int64(1), int64(1), object(9)
memory usage: 11.9+ MB


In [75]:
# =========================================================
# RANKING DE ORGANISMOS POR CANTIDAD DE LICITACIONES
# =========================================================

# Paso 1: Agrupar por organismo y contar licitaciones
df_ranking = (
    df_filtrado
    .groupby('codigo_organismo')
    .size()
    .reset_index(name='cantidad_licitaciones')
)

# Paso 2: Ordenar de mayor a menor
df_ranking = df_ranking.sort_values(by='cantidad_licitaciones', ascending=False)

# Paso 3: Crear ranking (manejo de empates tipo "dense")
df_ranking['ranking'] = (
    df_ranking['cantidad_licitaciones']
    .rank(method='dense', ascending=False)
    .astype(int)
)

# Paso 4: Calcular percentil (útil para segmentación futura)
df_ranking['percentil'] = (
    df_ranking['cantidad_licitaciones']
    .rank(pct=True)
)

# Paso 5: Orden final y selección de columnas
df_ranking = df_ranking[
    ['ranking', 'codigo_organismo', 'cantidad_licitaciones', 'percentil']
].sort_values(by='ranking')

# Mostrar resultado
display(df_ranking.head(20))

,ranking,codigo_organismo,cantidad_licitaciones,percentil
43,1,1075963,1570,1.000000
17,2,1057489,1369,0.997783
414,3,729,1198,0.995565
23,4,1057501,1176,0.993348
8,5,1057049,973,0.991131
392,6,5586,967,0.988914
36,7,1057547,963,0.986696
97,8,1725,941,0.984479
111,9,2080,891,0.982262
94,10,1658,882,0.980044
